<a href="https://colab.research.google.com/github/saqib0-cpu/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd

# reason code generator
def reason_code(row):
    if row['status'] == 'declining':
        return f"CTR/clicks dropped {abs(row['clicks_delta_pct']):.0%}, position worsened by {abs(row['position_delta']):.1f}"
    elif row['status'] == 'growing':
        return f"Clicks up {row['clicks_delta_pct']:.0%}, position improved by {row['position_delta']:.1f}"
    elif row['status'] == 'recovering':
        return f"Clicks up {row['clicks_delta_pct']:.0%} despite weaker position — organic recovery signal"
    elif row['status'] == 'stagnant':
        return f"Click change within ±10% — low CTR ({row['ctr_b']:.2%}) suggests metadata gap"
    else:
        return "Mixed/noisy signal — needs manual review"

features_df['reason_code'] = features_df.apply(reason_code, axis=1)

# use the model's predicted decline probability to rank urgency
features_df['decline_risk_score'] = model_grouped.predict_proba(X[feature_cols])[:, 1]

ranked_queue = features_df[[
    'content_hash_id', 'status', 'action', 'reason_code',
    'decline_risk_score', 'clicks_delta_pct', 'position_delta', 'ctr_b'
]].sort_values('decline_risk_score', ascending=False)

ranked_queue.head(20)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
Intended use: This playbook ranks content pages by estimated decline risk and suggests
an action (protect, rewrite, monitor, improve, review) to help a content team prioritize
where to spend manual review time first — it does not replace editorial judgment.

Limits:
- Trained and validated on FlyRank internship warehouse data (Aug 2025–Apr 2026) —
  not guaranteed to generalize to other time periods, verticals, or clients outside
  this dataset.
- The will_decline label uses a 15% click-drop threshold — a design choice, not a
  validated business rule.
- ctr_b dominates the model's decisions (~78% importance) — the ranking is largely a
  CTR-efficiency signal, not a holistic content-quality judgment.
- Measured F1 under a client-grouped split was [insert Week-6 number] — lower than the
  original random-split score, meaning real-world generalization is likely more modest
  than the headline number suggests.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
Human review required for:
- Any "rewrite" action before content is actually rewritten or retired
- Pages flagged "declining" with low search_volume (model has less reliable signal here)
- Any client-facing or legally sensitive content, regardless of score

NOT to be automated (no-go list):
- Auto-deleting or auto-unpublishing content based on decline_risk_score
- Auto-generating replacement content without human edit/approval
- Using this score as the sole input for client billing or performance conversations
- Applying this model to a clie


Monitoring:
- Track weekly what % of "declining" flagged pages actually saw continued click drops
  in the following window — measures real precision over time
- Track feature importance drift — if ctr_b's dominance changes significantly, the
  underlying pattern may have shifted

Retrain triggers:
- If observed precision on live data drops meaningfully below the measured 0.70 baseline
- Every quarter, regardless of performance, since search behavior/algorithms shift
- If a new content type or client vertical is added that wasn't in training data

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

ranked_queue.to_csv('work/outputs/ranked_action_queue.csv', index=False)

import json
metrics = {
    'baseline_f1_random_split': 0.154,
    'model_f1_random_split': 0.598,
    'model_f1_grouped_split': None,  # apna Week-6 actual number daalo
    'precision': 0.70,
    'recall': 0.52,
    'top_feature': 'ctr_b',
    'top_feature_importance': 0.779
}
with open('work/outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported ranked_action_queue.csv and metrics.json")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.